In [1]:
import torch
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 3050

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet201_Weights
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
import numpy as np
from tqdm import tqdm  # Progress bars

In [3]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
data_dir = '/home/rifat-cou/Documents/Project/Dataset_Raw'  # Your folder
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
class_names = full_dataset.classes  # For classification_report

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform

# Val loader
batch_size = 16  # For 6GB VRAM
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Bootstrapped train subsets (3 for bagging, same size as train with replacement)
num_bags = 3
bootstrap_indices = [np.random.choice(len(train_dataset), len(train_dataset), replace=True) for _ in range(num_bags)]
bootstrap_datasets = [Subset(train_dataset, indices) for indices in bootstrap_indices]

# Loaders for each bootstrap
train_loaders = [DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True) for ds in bootstrap_datasets]

num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
class_names = full_dataset.classes
print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

cuda
Classes: ['Chikenpox', 'Cowpox', 'Measles', 'MonkeyPox', 'Normal']
Train: 2088, Val: 523


In [4]:
def create_model():
    model = models.densenet201(weights=DenseNet201_Weights.DEFAULT)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    for param in model.features[:8].parameters():
        param.requires_grad = False
    return model.to(device)

# Create 3 models
models_list = [create_model() for _ in range(num_bags)]

In [5]:
def train_single_model(model, loader, epochs, save_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    best_acc = 0.0
    unfreeze_epoch = epochs // 2

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validate on full val (shared)
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(val_labels, val_preds)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {train_loss/len(loader):.4f} - Val Acc: {acc:.4f}')
        
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'{save_name}_best.pth')
        
        scheduler.step()

        if epoch == unfreeze_epoch:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)

# Train each bagged model
for i, (model, loader) in enumerate(zip(models_list, train_loaders)):
    train_single_model(model, loader, 300, f'bagged_densenet201_{i+1}')

100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  3.98it/s]


Epoch 1/300 - Loss: 0.4762 - Val Acc: 0.9082


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.07it/s]


Epoch 2/300 - Loss: 0.1039 - Val Acc: 0.9159


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.07it/s]


...

Epoch 298/300 - Loss: 0.0000 - Val Acc: 0.9273


100%|█████████████████████████████████████████| 131/131 [00:48<00:00,  2.71it/s]


Epoch 299/300 - Loss: 0.0000 - Val Acc: 0.9216


100%|█████████████████████████████████████████| 131/131 [00:48<00:00,  2.71it/s]


Epoch 300/300 - Loss: 0.0000 - Val Acc: 0.9293


In [10]:
# Load best models
for i, model in enumerate(models_list):
    model.load_state_dict(torch.load(f'bagged_densenet201_{i+1}_best.pth'))

def bagging_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1)

In [11]:
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = bagging_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

# Save
model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'V9_bagging_densenet201_x3')
os.makedirs(ensemble_dir, exist_ok=True)
for i, model in enumerate(models_list):
    torch.save(model.state_dict(), os.path.join(ensemble_dir, f'bagged_densenet201_{i+1}.pth'))

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [6]:
# Assuming models_list = [model1, model2, model3] are loaded (3 bagged DenseNet201)
def bagging_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = bagging_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

              precision    recall  f1-score   support

   Chikenpox       0.90      0.94      0.92       100
      Cowpox       1.00      0.97      0.99       102
     Measles       0.99      0.98      0.98        98
   MonkeyPox       0.93      0.91      0.92       124
      Normal       0.96      0.99      0.98        99

    accuracy                           0.96       523
   macro avg       0.96      0.96      0.96       523
weighted avg       0.96      0.96      0.96       523

Final Val Accuracy: 0.9560 - F1 Score: 0.9561


In [7]:
class BaggingEnsembleONNX(nn.Module):
    def __init__(self, base_models):
        super(BaggingEnsembleONNX, self).__init__()
        self.bases = nn.ModuleList(base_models)

    def forward(self, x):
        probs = [torch.softmax(base(x), dim=1) for base in self.bases]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
        return avg_probs  # Return probs (argmax in post-processing)

# Create combined model
bagging_onnx_model = BaggingEnsembleONNX(models_list)
bagging_onnx_model.eval()

BaggingEnsembleONNX(
  (bases): ModuleList(
    (0-2): 3 x DenseNet(
      (features): Sequential(
        (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu0): ReLU(inplace=True)
        (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (denseblock1): _DenseBlock(
          (denselayer1): _DenseLayer(
            (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu1): ReLU(inplace=True)
            (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu2): ReLU(inplace=True)
            (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          )
          (denselayer2): _Den

In [12]:
# Move to CPU for export
bagging_onnx_model.to('cpu')
for base in bagging_onnx_model.bases:
    base.to('cpu')

dummy_input = torch.randn(1, 3, 224, 224)  # Batch=1
model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'V9_bagging_densenet201_x3')
os.makedirs(ensemble_dir, exist_ok=True)
onnx_path = os.path.join(ensemble_dir, 'bagging_ensemble_v9.onnx')
torch.onnx.export(
    bagging_onnx_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f'ONNX exported: {onnx_path}')

# Optional verification
import onnxruntime as ort
ort_session = ort.InferenceSession(onnx_path)
ort_input = {ort_session.get_inputs()[0].name: dummy_input.numpy()}
ort_out = ort_session.run(None, ort_input)
print('ONNX verification passed')

ONNX exported: /home/rifat-cou/Documents/Project/Ensemble Models/V9_bagging_densenet201_x3/bagging_ensemble_v9.onnx
ONNX verification passed
